# Model Sanity Check
**Purpose:** Validate that the gold feature store and label store produced by `main.py` are ML-compatible.  

In [1]:
import os
import glob
import pandas as pd
import numpy as np
import pyspark
from pyspark.sql.functions import col

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

## 1. Setup Spark and load gold tables

In [2]:
# set path for Hadoop (Windows only, comment out on Mac/Linux)
os.environ["HADOOP_HOME"] = "C:\\hadoop"
os.environ["PATH"] = os.environ["PATH"] + ";C:\\hadoop\\bin"

# Initialize SparkSession
spark = pyspark.sql.SparkSession.builder \
    .appName("model_sanity_check") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

In [3]:
# Load gold feature store (all partitions)
gold_feature_store_directory = "datamart/gold/feature_store/"
feature_files = [gold_feature_store_directory + os.path.basename(f)
                 for f in glob.glob(os.path.join(gold_feature_store_directory, '*.parquet'))]
df_features = spark.read.option("header", "true").parquet(*feature_files)
print("Feature store row count:", df_features.count())

# Load gold label store (all partitions)
gold_label_store_directory = "datamart/gold/label_store/"
label_files = [gold_label_store_directory + os.path.basename(f)
               for f in glob.glob(os.path.join(gold_label_store_directory, '*.parquet'))]
df_labels = spark.read.option("header", "true").parquet(*label_files)
print("Label store row count:", df_labels.count())

Feature store row count: 12500
Label store row count: 12500


## 2. Join feature store and label store

In [4]:
# Join on loan_id
# Label store has: loan_id, Customer_ID, label, label_def, snapshot_date
# Feature store has: Customer_ID, loan_id, <features>, snapshot_date
# join on loan_id to get labels matched to features for the same loan

df_joined = df_features.join(
    df_labels.select("loan_id", "label"),
    on="loan_id",
    how="inner"
)
print("Joined dataset row count:", df_joined.count())
df_joined.groupBy("label").count().show()

Joined dataset row count: 12500
+-----+-----+
|label|count|
+-----+-----+
|    1| 3602|
|    0| 8898|
+-----+-----+



In [5]:
# Convert to pandas for sklearn
pdf = df_joined.toPandas()
print("Shape:", pdf.shape)
pdf.head()

Shape: (12500, 107)


,loan_id,Customer_ID,Age,Occupation,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,Num_Credit_Card,Interest_Rate,Num_of_Loan,...,fe_19_std,fe_19_min,fe_19_max,fe_20_mean,fe_20_std,fe_20_min,fe_20_max,has_clickstream,snapshot_date,label
0,CUS_0x1000_2023_05_01,CUS_0x1000,18.0,7.0,30625.939453,2706.161621,6.0,5.0,27.0,2.0,...,83.902920,4.0,200.0,111.2,80.334924,31.0,234.0,1,2023-05-01,1
1,CUS_0x108a_2023_05_01,CUS_0x108a,38.0,6.0,36982.359375,2554.813721,7.0,9.0,21.0,9.0,...,70.563447,17.0,184.0,177.4,190.700813,-88.0,416.0,1,2023-05-01,0
2,CUS_0x10f9_2023_05_01,CUS_0x10f9,54.0,NaN,150131.687500,11102.135742,5.0,1.0,4.0,0.0,...,82.240501,-7.0,221.0,52.0,84.080319,-45.0,147.0,1,2023-05-01,0
3,CUS_0x1119_2023_05_01,CUS_0x1119,36.0,5.0,56301.898438,4593.825195,9.0,5.0,26.0,2.0,...,77.961529,-64.0,125.0,86.2,50.251368,24.0,161.0,1,2023-05-01,1
4,CUS_0x1192_2023_05_01,CUS_0x1192,NaN,10.0,16319.375000,1518.947876,7.0,3.0,1.0,2.0,...,104.506938,-16.0,246.0,75.4,82.832964,9.0,211.0,1,2023-05-01,0


## 3. Prepare features and labels

In [6]:
# Define columns to drop (non-feature columns)
drop_cols = ["Customer_ID", "loan_id", "snapshot_date", "label"]

feature_cols = [c for c in pdf.columns if c not in drop_cols]
print(f"Number of features: {len(feature_cols)}")
print(f"Feature columns: {feature_cols}")

Number of features: 103
Feature columns: ['Age', 'Occupation', 'Annual_Income', 'Monthly_Inhand_Salary', 'Num_Bank_Accounts', 'Num_Credit_Card', 'Interest_Rate', 'Num_of_Loan', 'Delay_from_due_date', 'Num_of_Delayed_Payment', 'Changed_Credit_Limit', 'Num_Credit_Inquiries', 'Credit_Mix', 'Outstanding_Debt', 'Credit_Utilization_Ratio', 'Credit_History_Age', 'Payment_of_Min_Amount', 'Total_EMI_per_month', 'Amount_invested_monthly', 'Monthly_Balance', 'spending_level', 'payment_value_level', 'fe_1_mean', 'fe_1_std', 'fe_1_min', 'fe_1_max', 'fe_2_mean', 'fe_2_std', 'fe_2_min', 'fe_2_max', 'fe_3_mean', 'fe_3_std', 'fe_3_min', 'fe_3_max', 'fe_4_mean', 'fe_4_std', 'fe_4_min', 'fe_4_max', 'fe_5_mean', 'fe_5_std', 'fe_5_min', 'fe_5_max', 'fe_6_mean', 'fe_6_std', 'fe_6_min', 'fe_6_max', 'fe_7_mean', 'fe_7_std', 'fe_7_min', 'fe_7_max', 'fe_8_mean', 'fe_8_std', 'fe_8_min', 'fe_8_max', 'fe_9_mean', 'fe_9_std', 'fe_9_min', 'fe_9_max', 'fe_10_mean', 'fe_10_std', 'fe_10_min', 'fe_10_max', 'fe_11_mean',

In [7]:
X = pdf[feature_cols].copy()
y = pdf["label"].copy()

# Check for nulls
null_counts = X.isnull().sum()
print("Columns with nulls:")
print(null_counts[null_counts > 0])
print(f"\nTotal nulls: {X.isnull().sum().sum()}")
print(f"Label distribution:\n{y.value_counts()}")

Columns with nulls:
Age                   988
Occupation            880
Num_Bank_Accounts     171
Num_Credit_Card       296
Interest_Rate         270
                     ... 
fe_19_max            3526
fe_20_mean           3526
fe_20_std            4056
fe_20_min            3526
fe_20_max            3526
Length: 96, dtype: int64

Total nulls: 303608
Label distribution:
label
0    8898
1    3602
Name: count, dtype: int64


In [ ]:
# Simple null handling: fill with median for LR
X = X.fillna(X.median())

print("Nulls after fill:", X.isnull().sum().sum())
print("Shape:", X.shape)

Nulls after fill: 0
Shape: (12500, 103)


## 4. Train/test split and model training

In [9]:
# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train set: {X_train.shape[0]} rows")
print(f"Test set:  {X_test.shape[0]} rows")
print(f"Train label distribution:\n{y_train.value_counts(normalize=True)}")

Train set: 10000 rows
Test set:  2500 rows
Train label distribution:
label
0    0.7118
1    0.2882
Name: proportion, dtype: float64


In [10]:
# Fit a simple logistic regression
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train, y_train)

print("Model trained successfully!")

Model trained successfully!


## 5. Evaluate model

In [11]:
# Predict on test set
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1]

# Classification report
print("Classification Report:")
print(classification_report(y_test, y_pred))

# AUC
auc = roc_auc_score(y_test, y_pred_proba)
print(f"ROC AUC Score: {auc:.4f}")

# Confusion matrix
print(f"\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Classification Report:
              precision    recall  f1-score   support

           0       0.80      0.91      0.85      1780
           1       0.67      0.45      0.54       720

    accuracy                           0.78      2500
   macro avg       0.74      0.68      0.70      2500
weighted avg       0.77      0.78      0.76      2500

ROC AUC Score: 0.8043

Confusion Matrix:
[[1621  159]
 [ 394  326]]


In [13]:
# Final summary
print("=" * 50)
print("SANITY CHECK SUMMARY")
print("=" * 50)
print(f"Feature store rows:  {df_features.count()}")
print(f"Label store rows:    {df_labels.count()}")
print(f"Joined rows:         {pdf.shape[0]}")
print(f"Number of features:  {len(feature_cols)}")
print(f"ROC AUC:             {auc:.4f}")
print()


SANITY CHECK SUMMARY
Feature store rows:  12500
Label store rows:    12500
Joined rows:         12500
Number of features:  103
ROC AUC:             0.8043



Conclusion:
- The feature store is ML-compatible (all numeric, joinable to labels)
- The label store has valid binary labels
- There is no obvious data leakage (AUC should not be suspiciously close to 1.0)
- The pipeline output is ready for model training